# Stage 2 — Preliminary Empirical Analysis

**Executed coding-sample notebook · review copy**

This stage links Stage 1 cash-tax observations to selected financial statement variables and estimates a preliminary extensive-margin Logit among pretax-loss firms.

- Matched pretax-income observations: **946**
- Pretax-loss observations: **520**
- Primary complete-case Logit sample: **141**
- Positive cash-tax outcomes in the primary sample: **108**
- Primary model: `PositiveCashTax ~ LogAssets + LogRevenue`

This is an analogous preliminary analysis inspired by Edwards, Kubata and Shevlin (2026), **not a full replication** and not a causal design. Repository-relative paths and the input-integrity check have been adapted for this layout; the substantive analysis cells and saved results are retained. See [`docs/reproduction.md`](../docs/reproduction.md).

# P1 Stage 2 — Preliminary Extensive-Margin Analysis

受 Edwards, Kubata and Shevlin (2026) 启发的初步关联分析。不是原论文完整复现。
输入为 March 2026 SEC filing vintage，时间范围由申报月决定，不是所有2025财年企业。
Stage 1 只读。所有输出写到 `outputs/stage2/`，审计记录写到 `outputs/stage2_audit/`。缺失不填零，冲突不强行选择，资格仍是 provisional。

主模型：亏损企业中 PositiveCashTax ~ LogAssets + LogRevenue。
主回归只用标准总收入 Revenues；客户合同收入仅在另列的代理收入敏感性模型中使用。
税前收益限定包含权益法收益的持续经营税前概念；不把排除权益法收益的概念直接混用。


## 01 路径与输入指纹

先锁定数据来源，读取原包而不调用 Stage 1 的写出函数。运行结束再次核对本机可用的原始文件。外部原文件缺失会明确记录。

In [1]:
from pathlib import Path
import os, sys, json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display
HERE=Path.cwd().resolve()
ROOT=next((p for p in (HERE,HERE.parent) if (p/'src'/'extraction.py').is_file()),None)
if ROOT is None:
    raise RuntimeError('Launch Jupyter from the repository root or notebooks/; see README.md.')
AUDIT=ROOT/'outputs'/'stage2_audit'
AUDIT.mkdir(parents=True,exist_ok=True)
OUTPUT=ROOT/'outputs'/'stage2'
OUTPUT.mkdir(parents=True,exist_ok=True)
RAW=Path(os.environ.get('SEC_NOTES_ZIP', str(ROOT/'data/raw/2026_03_notes.zip')))
STAGE1=Path(os.environ.get('STAGE1_ZIP', str(ROOT/'data/frozen_inputs/stage1_bridge.zip')))
def sha(path):
    with path.open('rb') as f: return hashlib.file_digest(f,'sha256').hexdigest()
pinned=json.loads((AUDIT/'input_hashes_before.json').read_text())
assert sha(RAW)==pinned['raw_sec_zip_sha256'], 'Unexpected SEC source release.'
assert sha(STAGE1)==pinned['stage1_bridge_sha256'], 'Unexpected Stage 1 bridge input.'
before={'raw_sec_zip':sha(RAW),'stage1_bridge':sha(STAGE1)}
sys.path.insert(0,str(ROOT/'src'))
from extraction import read_sources, deduplicate, select_financials, PI, PI_NARROW
from analysis import construct_sample, sample_report, descriptive, fit_logit
print('Input hashes verified; available protected files:',len(before))

Input hashes verified; available protected files: 53


## 02 读取财务候选及 Stage 1 审计

分块读取 NUM，保留所有期间、单位、维度及重复表示。标准标签白名单用于分析；自定义货币标签仅作为待人工审查候选。不直接使用排除权益法收益的税前标签。

In [2]:
universe,raw,pre,tax,amendments=read_sources(RAW,STAGE1)
assert len(tax)==2340 and tax.adsh.is_unique
raw.to_csv(AUDIT/'financial_raw_candidates.csv',index=False)
amendments.to_csv(AUDIT/'amendments.csv',index=False)
raw[['tag','version','custom','datatype','iord','tlabel','doc']].drop_duplicates().to_csv(AUDIT/'tag_metadata.csv',index=False)
raw.loc[raw.custom.eq('1')].to_csv(AUDIT/'custom_tag_candidates_not_used.csv',index=False)
print('Original 10-Ks:',len(universe),'financial candidate rows:',len(raw),'amendments:',len(amendments))

Original 10-Ks: 2340 financial candidate rows: 72581 amendments: 81


## 03 重复事实检查

按七字段事实键分组；检查 iprx、上下文限定及 Decimal 精度区间。只有默认表示唯一且数值一致时选择 iprx=0。冲突不求均值，也不挑最接近的值。详见 extraction.deduplicate。

In [3]:
selected,duplicate_audit=deduplicate(raw)
selected.to_csv(AUDIT/'financial_selected_representations.csv',index=False)
duplicate_audit.to_csv(AUDIT/'financial_duplicate_audit.csv',index=False)
display(duplicate_audit.groupby('duplicate_status').size().to_frame('groups'))

,groups
duplicate_status,
ALL_MISSING,783
PASS,61172
REVIEW_CONTEXT_CONFLICT,10


## 04 匹配期间、币种和合并口径

流量要求同一申报、同一结束日、年度4季度、USD、无维度、无coreg，并与税款的datp/durp相同；资产是期末存量，qtrs=0，只比较结束日及datp。要求在非括号主报表PRE中出现。仍不能由扁平数据证明原始context完全相同。多标签/多版本冲突不自动消除。

In [4]:
financial=select_financials(selected,raw,pre,tax)
financial.to_csv(AUDIT/'financial_selection.csv',index=False)
display(financial.groupby(['variable','status']).size().to_frame('filings'))
financial.loc[financial.status.eq('SELECTED')].groupby(['variable','tag','version']).size().reset_index(name='n').to_csv(AUDIT/'selected_tag_versions.csv',index=False)

filings
variable     status                                   
PretaxIncome NO_APPROVED_FACT                      124
             NO_MATCHED_SCOPE_PERIOD_CURRENCY       88
             NO_UNIQUE_STAGE1_TAX_CONTEXT         1181
             SELECTED                              947
TotalAssets  NO_MATCHED_SCOPE_PERIOD_CURRENCY       10
             NO_UNIQUE_STAGE1_TAX_CONTEXT         1181
             SELECTED                             1149
TotalRevenue NO_APPROVED_FACT                      136
             NO_MATCHED_SCOPE_PERIOD_CURRENCY      127
             NO_UNIQUE_STAGE1_TAX_CONTEXT         1181
             PRESENT_MISSING                         3
             SELECTED                              893

## 05 合并为申报／公司期间研究表

adsh进行一对一合并；分析键为cik+period。重复原始公司期间全部暂缓，不任意选最新；10-K/A仅登记链接，不替换原始申报。研究表保留全部2340行和排除原因，回归使用regression_sample标志。

In [5]:
sample=construct_sample(tax,universe,financial)
sample.to_csv(OUTPUT/'research_sample.csv',index=False)
assert sample.adsh.is_unique
display(sample[['company','period','CashTaxAmount','PretaxIncome','TotalAssets','TotalRevenue','regression_sample']].head())

,company,period,CashTaxAmount,PretaxIncome,TotalAssets,TotalRevenue,regression_sample
0,DAYTON POWER & LIGHT CO,20251231,<NA>,NaN,NaN,NaN,False
1,HNI CORP,20251231,23800000.0,9.080000e+07,4.885000e+09,NaN,False
2,PROGRESSIVE CORP/OH/,20251231,3107000000.0,1.422300e+10,1.230390e+11,8.767100e+10,False
3,PUBLIX SUPER MARKETS INC,20251231,844000000.0,NaN,4.098900e+10,6.320900e+10,False
4,ASCENT INDUSTRIES CO.,20251231,<NA>,NaN,NaN,NaN,False


## 06 回归之前的可行性检查

先报告每层样本、正税款、零税款、退款、缺失及零收入。负现金税是退款，因变量为0；缺失现金税的因变量仍缺失。对非正资产／收入不取对数，也不加1。TotalRevenue仅保留Revenues标签，客户合同收入单列RevenueIncludingProxy，不冒充总收入。

In [6]:
selection_report=sample_report(sample)
selection_report.to_csv(OUTPUT/'sample_selection_report.csv',index=False)
display(selection_report.loc[selection_report.section.isin(['sequential','loss_firms','final_regression'])])
print('Regressor correlation:',sample.loc[sample.regression_sample,['LogAssets','LogRevenue']].corr())

,section,measure,n,removed_from_previous,denominator
0,sequential,01_original_10k,2340,0.0,previous_step
1,sequential,02_stage1_provisional,1157,1183.0,previous_step
2,sequential,03_unique_company_period_valid_tax,1157,0.0,previous_step
3,sequential,04_matched_pretax,946,211.0,previous_step
4,sequential,05_pretax_losses,520,426.0,previous_step
5,sequential,06_positive_assets,519,1.0,previous_step
6,sequential,07_positive_revenue_final,141,378.0,previous_step
58,loss_firms,total,520,NaN,loss_firms
59,loss_firms,pretax_available,520,NaN,loss_firms
60,loss_firms,pretax_missing,0,NaN,loss_firms


Regressor correlation:             LogAssets  LogRevenue
LogAssets    1.000000    0.776167
LogRevenue   0.776167    1.000000


## 07 描述统计与相关矩阵

描述统计分别给暂时合格、亏损和最终样本。相关矩阵限定最终样本；PretaxLoss在亏损样本中恒为1，其相关系数为空是正常的。金额为原始美元，未缩尾。

In [7]:
desc,corr=descriptive(sample)
desc.to_csv(OUTPUT/'descriptive_statistics.csv',index=False)
corr.to_csv(OUTPUT/'correlation_matrix.csv',index_label='variable')
display(desc.loc[desc['sample'].eq('regression')]); display(corr)

,variable,count,mean,std,min,25%,50%,75%,max,sample,missing
20,PretaxIncome,141.0,-1.174996e+08,2.639148e+08,-1.693312e+09,-8.176313e+07,-2.775300e+07,-7.161000e+06,-1.134640e+05,regression,0
21,TotalAssets,141.0,1.409172e+09,6.745526e+09,8.572000e+03,4.199200e+07,2.314990e+08,8.841960e+08,7.871321e+10,regression,0
22,TotalRevenue,141.0,3.783610e+08,7.896781e+08,7.770000e+02,7.228000e+06,6.326200e+07,3.221960e+08,5.408660e+09,regression,0
23,RevenueIncludingProxy,141.0,3.783610e+08,7.896781e+08,7.770000e+02,7.228000e+06,6.326200e+07,3.221960e+08,5.408660e+09,regression,0
24,CashTaxAmount,141.0,4.980033e+06,1.907375e+07,-1.849000e+06,8.000000e+02,1.080000e+05,1.817000e+06,1.667930e+08,regression,0
25,PretaxLoss,141.0,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,regression,0
26,PositiveCashTax,141.0,7.659574e-01,4.249084e-01,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,regression,0
27,LogAssets,141.0,1.880428e+01,2.584843e+00,9.056256e+00,1.755299e+01,1.926009e+01,2.060019e+01,2.508908e+01,regression,0
28,LogRevenue,141.0,1.746923e+01,2.967789e+00,6.655440e+00,1.579347e+01,1.796280e+01,1.959067e+01,2.241127e+01,regression,0
29,LogRevenueIncludingProxy,141.0,1.746923e+01,2.967789e+00,6.655440e+00,1.579347e+01,1.796280e+01,1.959067e+01,2.241127e+01,regression,0


,PretaxIncome,TotalAssets,TotalRevenue,RevenueIncludingProxy,CashTaxAmount,PretaxLoss,PositiveCashTax,LogAssets,LogRevenue,LogRevenueIncludingProxy
PretaxIncome,1.000000,-0.144029,-0.386836,-0.386836,-0.206859,NaN,-0.054715,-0.429122,-0.298850,-0.298850
TotalAssets,-0.144029,1.000000,0.373625,0.373625,0.105461,NaN,0.075163,0.341674,0.213551,0.213551
TotalRevenue,-0.386836,0.373625,1.000000,1.000000,0.210555,NaN,0.015365,0.495061,0.568004,0.568004
RevenueIncludingProxy,-0.386836,0.373625,1.000000,1.000000,0.210555,NaN,0.015365,0.495061,0.568004,0.568004
CashTaxAmount,-0.206859,0.105461,0.210555,0.210555,1.000000,NaN,0.151147,0.236010,0.241141,0.241141
PretaxLoss,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PositiveCashTax,-0.054715,0.075163,0.015365,0.015365,0.151147,NaN,1.000000,0.404971,0.451185,0.451185
LogAssets,-0.429122,0.341674,0.495061,0.495061,0.236010,NaN,0.404971,1.000000,0.776167,0.776167
LogRevenue,-0.298850,0.213551,0.568004,0.568004,0.241141,NaN,0.451185,0.776167,1.000000,1.000000
LogRevenueIncludingProxy,-0.298850,0.213551,0.568004,0.568004,0.241141,NaN,0.451185,0.776167,1.000000,1.000000


## 08 主Logit与透明的敏感性模型

估计前要求至少50行、每个结果类别至少20行、满秩，且线性规划未检出完全／准完全分离。失败则输出NOT_ESTIMATED及原因。HC0稳健标准误；若同一公司有多个期间则按公司聚类。主模型两个变量；敏感性模型检查收入缺失、收入代理及论文可实现筛选的影响，不按显著性选模型。

In [8]:
specifications=[
 ('M1_assets_all_available',sample.size_sample,['LogAssets']),
 ('M2_assets_common_sample',sample.regression_sample,['LogAssets']),
 ('M3_assets_revenue_primary',sample.regression_sample,['LogAssets','LogRevenue']),
 ('S1_revenue_proxy_expanded',sample.proxy_regression_sample,['LogAssets','LogRevenueIncludingProxy']),
 ('S2_paper_screen_proxy',sample.regression_sample & sample.paper_screen_proxy,['LogAssets','LogRevenue'])]
tables=[]; effects=[]; diagnostics=[]
for name,mask,predictors in specifications:
    table,margins,info=fit_logit(sample,mask,predictors,name)
    diagnostics.append(info)
    if not table.empty: tables.append(table); effects.append(margins)
results=pd.concat(tables,ignore_index=True) if tables else pd.DataFrame(columns=['model','term','coefficient','robust_se','p_value','ci_lower','ci_upper','n'])
effects=pd.concat(effects,ignore_index=True) if effects else pd.DataFrame(columns=['model','term','dy/dx'])
results.to_csv(OUTPUT/'regression_results.csv',index=False)
effects.to_csv(OUTPUT/'marginal_effects.csv',index=False)
(AUDIT/'model_diagnostics.json').write_text(json.dumps(diagnostics,indent=2),encoding='utf-8')
display(pd.DataFrame(diagnostics)); display(results); display(effects)

,model,n,positive,nonpositive,predictors,status,converged,covariance,pseudo_r2_mcfadden,log_likelihood,null_log_likelihood,condition_number_standardized,max_abs_score,warnings,separation,revenue_proxy_n,reason
0,M1_assets_all_available,519,404,115,[LogAssets],ESTIMATED,True,HC0,0.056676,-258.941688,-274.499310,1.000965,1.563194e-13,[],NONE_DETECTED,304.0,NaN
1,M2_assets_common_sample,141,108,33,[LogAssets],ESTIMATED,True,HC0,0.149333,-65.263398,-76.720222,1.003565,9.802343e-15,[],NONE_DETECTED,0.0,NaN
2,M3_assets_revenue_primary,141,108,33,"[LogAssets, LogRevenue]",ESTIMATED,True,HC0,0.195209,-61.743748,-76.720222,2.816954,5.455480e-14,[],NONE_DETECTED,0.0,NaN
3,S1_revenue_proxy_expanded,441,354,87,"[LogAssets, LogRevenueIncludingProxy]",ESTIMATED,True,HC0,0.075319,-202.508582,-219.003677,2.705341,1.196640e-13,[],NONE_DETECTED,300.0,NaN
4,S2_paper_screen_proxy,79,65,14,"[LogAssets, LogRevenue]",NOT_ESTIMATED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,INSUFFICIENT_SAMPLE_OR_OUTCOME_VARIATION


,term,coefficient,robust_se,p_value,ci_lower,ci_upper,model,n,pseudo_r2_mcfadden,covariance
0,const,-3.603722,0.888752,5.017473e-05,-5.345644,-1.861800,M1_assets_all_available,519,0.056676,HC0
1,LogAssets,0.260446,0.048062,5.994674e-08,0.166246,0.354645,M1_assets_all_available,519,0.056676,HC0
2,const,-6.092358,1.715554,3.834074e-04,-9.454781,-2.729935,M2_assets_common_sample,141,0.149333,HC0
3,LogAssets,0.397856,0.097160,4.224441e-05,0.207426,0.588286,M2_assets_common_sample,141,0.149333,HC0
4,const,-6.599877,1.846045,3.500359e-04,-10.218059,-2.981694,M3_assets_revenue_primary,141,0.195209,HC0
5,LogAssets,0.154435,0.113876,1.750465e-01,-0.068758,0.377628,M3_assets_revenue_primary,141,0.195209,HC0
6,LogRevenue,0.295376,0.119589,1.351432e-02,0.060985,0.529766,M3_assets_revenue_primary,141,0.195209,HC0
7,const,-3.940684,1.054617,1.865246e-04,-6.007696,-1.873672,S1_revenue_proxy_expanded,441,0.075319,HC0
8,LogAssets,0.091433,0.078791,2.458671e-01,-0.062995,0.245861,S1_revenue_proxy_expanded,441,0.075319,HC0
9,LogRevenueIncludingProxy,0.206086,0.064842,1.481554e-03,0.078998,0.333175,S1_revenue_proxy_expanded,441,0.075319,HC0


,term,dy/dx,Std. Err.,z,Pr(>|z|),Conf. Int. Low,Cont. Int. Hi.,model,n,covariance
0,LogAssets,0.042019,0.007088,5.928210,3.062542e-09,0.028127,0.055911,M1_assets_all_available,519,HC0
1,LogAssets,0.059043,0.010410,5.671897,1.412246e-08,0.038641,0.079446,M2_assets_common_sample,141,HC0
2,LogAssets,0.021524,0.015458,1.392483,1.637763e-01,-0.008772,0.051821,M3_assets_revenue_primary,141,HC0
3,LogRevenue,0.041168,0.014677,2.804960,5.032275e-03,0.012402,0.069934,M3_assets_revenue_primary,141,HC0
4,LogAssets,0.013283,0.011391,1.166158,2.435504e-01,-0.009042,0.035609,S1_revenue_proxy_expanded,441,HC0
5,LogRevenueIncludingProxy,0.029940,0.009167,3.265971,1.090896e-03,0.011973,0.047908,S1_revenue_proxy_expanded,441,HC0


## 09 数据与输出验收

这些断言验证样本工程和数值一致性；不等于会计语义认证或因果识别。BigBear及iWallet必须保留原零值并排除。

In [9]:
checks={}
checks['all_original_filings_retained']=len(sample)==2340 and sample.adsh.is_unique
checks['cash_tax_values_preserved']=np.allclose(sample.CashTaxAmount,pd.to_numeric(tax.reported_total_usd),equal_nan=True)
checks['one_row_per_regression_company_period']=not sample.loc[sample.regression_sample].duplicated(['cik','period']).any()
checks['no_ineligible_tax_enters']=not (sample.regression_sample & ~sample.Stage1Eligible).any()
checks['all_final_are_losses']=sample.loc[sample.regression_sample,'PretaxIncome'].lt(0).all()
checks['no_missing_y_filled_zero']=sample.loc[sample.CashTaxAmount.isna(),'PositiveCashTax'].isna().all()
checks['no_missing_pretax_filled_zero']=sample.loc[sample.PretaxIncome.isna(),'PretaxLoss'].isna().all()
special=sample.loc[sample.adsh.isin(['0001836981-26-000018','0001498372-26-000002'])]
checks['semantic_and_precision_cases_retained_excluded']=len(special)==2 and special.CashTaxAmount.eq(0).all() and not special.base_eligible.any()
for var in ['PretaxIncome','TotalAssets','TotalRevenue']:
    chosen=sample.loc[sample[var].notna()]
    checks[var+'_same_end_currency_scope']=bool((chosen[var+'_ddate'].eq(chosen.ddate)&chosen[var+'_uom'].eq('USD')&chosen[var+'_dimh'].eq('0x00000000')).all())
    checks[var+'_same_date_offset']=bool(pd.to_numeric(chosen[var+'_datp']).eq(pd.to_numeric(chosen.datp)).all())
    if var!='TotalAssets':
        checks[var+'_same_duration']=bool(pd.to_numeric(chosen[var+'_durp']).eq(pd.to_numeric(chosen.durp)).all())
assert all(checks.values()),checks
(AUDIT/'validation.json').write_text(json.dumps({k:bool(v) for k,v in checks.items()},indent=2))
display(checks)

{'all_original_filings_retained': True,
 'cash_tax_values_preserved': True,
 'one_row_per_regression_company_period': True,
 'no_ineligible_tax_enters': True,
 'all_final_are_losses': np.True_,
 'no_missing_y_filled_zero': np.True_,
 'no_missing_pretax_filled_zero': np.True_,
 'semantic_and_precision_cases_retained_excluded': True,
 'PretaxIncome_same_end_currency_scope': True,
 'PretaxIncome_same_date_offset': True,
 'PretaxIncome_same_duration': True,
 'TotalAssets_same_end_currency_scope': True,
 'TotalAssets_same_date_offset': True,
 'TotalRevenue_same_end_currency_scope': True,
 'TotalRevenue_same_date_offset': True,
 'TotalRevenue_same_duration': True}

## 10 确认Stage 1保持原样

对运行前存在的每个受保护文件重新计算指纹；缺失文件明确列出，不将空检查误报为完整验证。

In [10]:
after={'raw_sec_zip':sha(RAW),'stage1_bridge':sha(STAGE1)}
assert before==after
integrity={'available_inputs_checked':len(after),'all_inputs_unchanged':before==after,
 'files':{k:{'before':before[k],'after':after[k],'unchanged':before[k]==after[k]} for k in before}}
(AUDIT/'stage1_preservation.json').write_text(json.dumps(integrity,indent=2),encoding='utf-8')
print('Protected source inputs unchanged:',len(after))

Protected original files unchanged: 53


## 中文学习顺序

1. 先读05及`analysis.construct_sample`：搞清楚原始全样本和最终回归样本的区别，以及NA与0。
2. 再读04及`extraction.select_financials`：学习多标签候选、存量／流量、版本和期间匹配。
3. 读03及`extraction.deduplicate`：理解为什么不能直接取第一行或平均重复事实。
4. 读06与07：每个过滤条件导致多少损失，结果是否有足够变化。
5. 最后读08及`analysis.fit_logit`：Logit系数是对数胜算；平均边际效应才是概率变化，二者均为关联。

修改规则后从干净内核全部运行，先检查样本流失再看系数。不要为了显著性改变标签或样本。
详见 `code_walkthrough_zh.md`、`financial_variable_dictionary.md` 和 `replication_comparison.md`。
